# 1

In [ ]:
from dotenv import load_dotenv
import pymysql
import os
import pandas as pd

load_dotenv()

class MySQL_Doing:
    def __init__(self):
        self.Host = os.getenv("Host", "127.0.0.1")
        self.User = os.getenv("User", "root")
        self.Port = int(os.getenv("Port", 3308))   # ← 改這裡確保對應 port
        self.Password = os.getenv("Password_SQL", "")
        self.Database = os.getenv("Database", "")
        self.conn = None

    def _connect(self):
        return pymysql.connect(
            host=self.Host,
            user=self.User,
            port=self.Port,
            password=self.Password,
            database=self.Database,
            charset="utf8mb4",
            cursorclass=pymysql.cursors.DictCursor,
            autocommit=True,
            ssl_disabled=True   # ← 這一行關閉 SSL
        )

    def run(self, sql, params=None):
        try:
            conn = self._connect()
            with conn.cursor() as cursor:
                cursor.execute(sql, params or ())
                conn.commit()
                if cursor.description:
                    rows = cursor.fetchall()
                    return pd.DataFrame(rows)
                return None
        finally:
            try:
                conn.close()
            except:
                pass

# === 測試 ===
if __name__ == "__main__":
    db = MySQL_Doing()
    # df = db.run("SHOW TABLES;")
    df = db.run('select stop_name, schedule, direction from bus_route_stations where route_id = 2 and direction = "回程" order by stop_order;')
    # df = db.run('desc bus_route_stations;')
    print(df)

In [37]:
df = db.run('select stop_name, schedule, direction from bus_route_stations where route_id = 3 and direction = "去程" order by stop_order;')
print(df)

    stop_name                                           schedule direction
0       花蓮轉運站  17:45:00,18:00:00,18:20:00,18:30:00,18:50:00,1...        去程
1        勤天商旅  17:46:00,18:01:00,18:21:00,18:31:00,18:51:00,1...        去程
2       市立圖書館  17:49:00,18:04:00,18:24:00,18:34:00,18:54:00,1...        去程
3        麗星飯店  17:51:00,18:06:00,18:26:00,18:36:00,18:56:00,1...        去程
4   花蓮又一村文創園區  17:59:00,18:14:00,18:34:00,18:44:00,19:04:00,1...        去程
5      花蓮文創園區  18:01:00,18:16:00,18:36:00,18:46:00,19:06:00,1...        去程
6     日出香榭大道1                                               None        去程
7     日出香榭大道2                                               None        去程
8       東大門夜市  18:06:00,18:21:00,18:41:00,18:51:00,19:11:00,1...        去程
9         中華路  18:09:00,18:24:00,18:44:00,18:54:00,19:14:00,1...        去程
10      將軍府商圈  18:13:00,18:28:00,18:48:00,18:58:00,19:18:00,1...        去程
11       勤天商旅  17:46:00,18:01:00,18:21:00,18:31:00,18:51:00,1...        去程
12      花蓮轉運站  17:45:00,1

In [34]:
import pandas as pd

def Check(Numbver):
    excel_path = "到站時刻.xlsx"
    xls = pd.ExcelFile(excel_path)

    # === 顯示有哪些工作表 ===
    # print("工作表列表：")
    # for idx, name in enumerate(xls.sheet_names, start=1):
    #     print(f"{idx}. {name}")

    # === 使用者選擇 ===
    # choice = input(f"\n請輸入要查看的工作表編號 (1~{len(xls.sheet_names)}): ")
    choice = Numbver
    try:
        index = int(choice) - 1
        if index < 0 or index >= len(xls.sheet_names):
            raise ValueError("超出範圍")

        df = pd.read_excel(xls, sheet_name=xls.sheet_names[index])
        print(f"\n你選擇的是第 {index + 1} 個工作表：{xls.sheet_names[index]}")
        print(df.head())

    except ValueError as e:
        print("❌ 輸入錯誤，請輸入正確數字。", e)

Check(5)


你選擇的是第 5 個工作表：市民小巴7Go
   平日班次     花蓮轉運站      勤天商旅     市立圖書館      麗星飯店 花蓮又一村文創園區    花蓮文創園區     日出大道1  \
0     1  17:45:00  17:46:00  17:49:00  17:51:00  17:59:00  18:01:00  18:03:00   
1     2  18:00:00  18:01:00  18:04:00  18:06:00  18:14:00  18:16:00  18:18:00   
2     3  18:20:00  18:21:00  18:24:00  18:26:00  18:34:00  18:36:00  18:38:00   
3     4  18:30:00  18:31:00  18:34:00  18:36:00  18:44:00  18:46:00  18:48:00   
4     5  18:50:00  18:51:00  18:54:00  18:56:00  19:04:00  19:06:00  19:08:00   

      日出大道2     東大門夜市       中華路     將軍府商圈    勤天商旅.1   花蓮轉運站.1  
0  18:04:00  18:06:00  18:09:00  18:13:00  18:19:00  18:20:00  
1  18:19:00  18:21:00  18:24:00  18:28:00  18:34:00  18:35:00  
2  18:39:00  18:41:00  18:44:00  18:48:00  18:54:00  18:55:00  
3  18:49:00  18:51:00  18:54:00  18:58:00  19:04:00  19:05:00  
4  19:09:00  19:11:00  19:14:00  19:18:00  19:24:00  19:25:00  


In [35]:
import pandas as pd

excel_path = "到站時刻.xlsx"
sheet_name = "市民小巴7Go"

df_excel = pd.read_excel(excel_path, sheet_name=sheet_name)

# Excel 第一列通常是「返程、門諾醫院、中美民權八街、...」
# 我們要取每一站的時間欄（排除第一欄 "返程"）
time_map = {}

for col in df_excel.columns[1:]:
    # 將該站所有班次的時間串起來，用逗號分隔
    times = df_excel[col].dropna().astype(str).str.strip().tolist()
    time_map[col.strip()] = ",".join(times)

print("對應表：")
for k, v in time_map.items():
    print(k, "→", v)


對應表：
花蓮轉運站 → 17:45:00,18:00:00,18:20:00,18:30:00,18:50:00,19:15:00,19:30:00,20:00:00,20:30:00
勤天商旅 → 17:46:00,18:01:00,18:21:00,18:31:00,18:51:00,19:16:00,19:31:00,20:01:00,20:31:00
市立圖書館 → 17:49:00,18:04:00,18:24:00,18:34:00,18:54:00,19:19:00,19:34:00,20:04:00,20:34:00
麗星飯店 → 17:51:00,18:06:00,18:26:00,18:36:00,18:56:00,19:21:00,19:36:00,20:06:00,20:36:00
花蓮又一村文創園區 → 17:59:00,18:14:00,18:34:00,18:44:00,19:04:00,19:29:00,19:44:00,20:14:00,20:44:00
花蓮文創園區 → 18:01:00,18:16:00,18:36:00,18:46:00,19:06:00,19:31:00,19:46:00,20:16:00,20:46:00
日出大道1 → 18:03:00,18:18:00,18:38:00,18:48:00,19:08:00,19:33:00,19:48:00,20:18:00,20:48:00
日出大道2 → 18:04:00,18:19:00,18:39:00,18:49:00,19:09:00,19:34:00,19:49:00,20:19:00,20:49:00
東大門夜市 → 18:06:00,18:21:00,18:41:00,18:51:00,19:11:00,19:36:00,19:51:00,20:21:00,20:51:00
中華路 → 18:09:00,18:24:00,18:44:00,18:54:00,19:14:00,19:39:00,19:54:00,20:24:00,20:54:00
將軍府商圈 → 18:13:00,18:28:00,18:48:00,18:58:00,19:18:00,19:43:00,19:58:00,20:28:00,20:58:00
勤天商旅.1 → 18:19:

In [36]:
db = MySQL_Doing()

for stop, schedule_str in time_map.items():
    sql = """
        UPDATE bus_route_stations
        SET schedule = %s
        WHERE route_id = %s AND direction = %s AND stop_name = %s
    """
    params = (schedule_str, 3, "去程", stop)
    db.run(sql, params)
    print(f"✅ 更新完成：{stop}")

✅ 更新完成：花蓮轉運站
✅ 更新完成：勤天商旅
✅ 更新完成：市立圖書館
✅ 更新完成：麗星飯店
✅ 更新完成：花蓮又一村文創園區
✅ 更新完成：花蓮文創園區
✅ 更新完成：日出大道1
✅ 更新完成：日出大道2
✅ 更新完成：東大門夜市
✅ 更新完成：中華路
✅ 更新完成：將軍府商圈
✅ 更新完成：勤天商旅.1
✅ 更新完成：花蓮轉運站.1


In [33]:
df_check = db.run("""
    SELECT stop_name, schedule
    FROM bus_route_stations
    WHERE route_id = 2 AND direction = '回程'
    ORDER BY stop_order;
""")
print(df_check)


    stop_name                                           schedule
0        門諾醫院  09:35:00,10:25:00,11:15:00,12:05:00,12:55:00,1...
1      中美民權八街  09:37:00,10:27:00,11:17:00,12:07:00,12:57:00,1...
2        煙波飯店                                               None
3        璽濱飯店                                               None
4  花蓮醫院(慈愛大樓)  09:44:00,10:34:00,11:24:00,12:14:00,13:04:00,1...
5        明禮國小  09:47:00,10:37:00,11:27:00,12:17:00,13:07:00,1...
6        市民廣場  09:48:00,10:38:00,11:28:00,12:18:00,13:08:00,1...
7       花蓮轉運站  09:54:00,10:44:00,11:34:00,12:24:00,13:14:00,1...


# 2

In [5]:
import pandas as pd

# === Excel ===
excel_path = "到站時刻.xlsx"
sheet_name = "市民小巴7Go"   # ← 改成你要的表名
route_id = 3
direction = "去程"

# === 讀取 Excel ===
df = pd.read_excel(excel_path, sheet_name=sheet_name)

# === 生成對應字典 ===
updates = []
for col in df.columns[1:]:
    times = df[col].dropna().astype(str).str.strip().tolist()
    schedule_str = ",".join(times)
    stop_name = col.strip()
    sql = (
        f"UPDATE bus_route_stations "
        f"SET schedule = '{schedule_str}' "
        f"WHERE route_id = {route_id} "
        f"AND direction = '{direction}' "
        f"AND stop_name = '{stop_name}';"
    )
    updates.append(sql)

# === 輸出結果 ===
for s in updates:
    print(s)


UPDATE bus_route_stations SET schedule = '17:45:00,18:00:00,18:20:00,18:30:00,18:50:00,19:15:00,19:30:00,20:00:00,20:30:00' WHERE route_id = 3 AND direction = '去程' AND stop_name = '花蓮轉運站';
UPDATE bus_route_stations SET schedule = '17:46:00,18:01:00,18:21:00,18:31:00,18:51:00,19:16:00,19:31:00,20:01:00,20:31:00' WHERE route_id = 3 AND direction = '去程' AND stop_name = '勤天商旅';
UPDATE bus_route_stations SET schedule = '17:49:00,18:04:00,18:24:00,18:34:00,18:54:00,19:19:00,19:34:00,20:04:00,20:34:00' WHERE route_id = 3 AND direction = '去程' AND stop_name = '市立圖書館';
UPDATE bus_route_stations SET schedule = '17:51:00,18:06:00,18:26:00,18:36:00,18:56:00,19:21:00,19:36:00,20:06:00,20:36:00' WHERE route_id = 3 AND direction = '去程' AND stop_name = '麗星飯店';
UPDATE bus_route_stations SET schedule = '17:59:00,18:14:00,18:34:00,18:44:00,19:04:00,19:29:00,19:44:00,20:14:00,20:44:00' WHERE route_id = 3 AND direction = '去程' AND stop_name = '花蓮又一村文創園區';
UPDATE bus_route_stations SET schedule = '18:01:00,18

In [10]:
import pandas as pd

# === Excel 來源 ===
excel_path = "到站時刻.xlsx"
sheet_name = "市民小巴7Go"   # ← 這裡換成你實際的工作表名
route_id = 3
direction = "去程"

# === 讀 Excel ===
df = pd.read_excel(excel_path, sheet_name=sheet_name)

# === 每欄轉成 SQL ===
updates = []
for i, col in enumerate(df.columns[1:], start=1):
    times = df[col].dropna().astype(str).str.strip().tolist()
    schedule_str = ",".join(times)
    stop_name = col.strip()
    # 用 stop_order 區分（對應順序 i）
    sql = (
        f"UPDATE bus_route_stations "
        f"SET schedule = '{schedule_str}' "
        f"WHERE route_id = {route_id} "
        f"AND direction = '{direction}' "
        f"AND stop_order = {i};"
    )
    updates.append(sql)

# === 輸出 ===
for s in updates:
    print(s)


UPDATE bus_route_stations SET schedule = '17:45:00,18:00:00,18:20:00,18:30:00,18:50:00,19:15:00,19:30:00,20:00:00,20:30:00' WHERE route_id = 3 AND direction = '去程' AND stop_order = 1;
UPDATE bus_route_stations SET schedule = '17:46:00,18:01:00,18:21:00,18:31:00,18:51:00,19:16:00,19:31:00,20:01:00,20:31:00' WHERE route_id = 3 AND direction = '去程' AND stop_order = 2;
UPDATE bus_route_stations SET schedule = '17:49:00,18:04:00,18:24:00,18:34:00,18:54:00,19:19:00,19:34:00,20:04:00,20:34:00' WHERE route_id = 3 AND direction = '去程' AND stop_order = 3;
UPDATE bus_route_stations SET schedule = '17:51:00,18:06:00,18:26:00,18:36:00,18:56:00,19:21:00,19:36:00,20:06:00,20:36:00' WHERE route_id = 3 AND direction = '去程' AND stop_order = 4;
UPDATE bus_route_stations SET schedule = '17:59:00,18:14:00,18:34:00,18:44:00,19:04:00,19:29:00,19:44:00,20:14:00,20:44:00' WHERE route_id = 3 AND direction = '去程' AND stop_order = 5;
UPDATE bus_route_stations SET schedule = '18:01:00,18:16:00,18:36:00,18:46:00,19

In [8]:
UPDATE bus_route_stations SET schedule = '17:45:00,18:00:00,18:20:00,18:30:00,18:50:00,19:15:00,19:30:00,20:00:00,20:30:00' WHERE route_id = 3 AND direction = '去程' AND stop_order = 1;
UPDATE bus_route_stations SET schedule = '17:46:00,18:01:00,18:21:00,18:31:00,18:51:00,19:16:00,19:31:00,20:01:00,20:31:00' WHERE route_id = 3 AND direction = '去程' AND stop_order = 2;
UPDATE bus_route_stations SET schedule = '17:49:00,18:04:00,18:24:00,18:34:00,18:54:00,19:19:00,19:34:00,20:04:00,20:34:00' WHERE route_id = 3 AND direction = '去程' AND stop_order = 3;
UPDATE bus_route_stations SET schedule = '17:51:00,18:06:00,18:26:00,18:36:00,18:56:00,19:21:00,19:36:00,20:06:00,20:36:00' WHERE route_id = 3 AND direction = '去程' AND stop_order = 4;
UPDATE bus_route_stations SET schedule = '17:59:00,18:14:00,18:34:00,18:44:00,19:04:00,19:29:00,19:44:00,20:14:00,20:44:00' WHERE route_id = 3 AND direction = '去程' AND stop_order = 5;
UPDATE bus_route_stations SET schedule = '18:01:00,18:16:00,18:36:00,18:46:00,19:06:00,19:31:00,19:46:00,20:16:00,20:46:00' WHERE route_id = 3 AND direction = '去程' AND stop_order = 6;
UPDATE bus_route_stations SET schedule = '18:03:00,18:18:00,18:38:00,18:48:00,19:08:00,19:33:00,19:48:00,20:18:00,20:48:00' WHERE route_id = 3 AND direction = '去程' AND stop_order = 7;
UPDATE bus_route_stations SET schedule = '18:04:00,18:19:00,18:39:00,18:49:00,19:09:00,19:34:00,19:49:00,20:19:00,20:49:00' WHERE route_id = 3 AND direction = '去程' AND stop_order = 8;
UPDATE bus_route_stations SET schedule = '18:06:00,18:21:00,18:41:00,18:51:00,19:11:00,19:36:00,19:51:00,20:21:00,20:51:00' WHERE route_id = 3 AND direction = '去程' AND stop_order = 9;
UPDATE bus_route_stations SET schedule = '18:09:00,18:24:00,18:44:00,18:54:00,19:14:00,19:39:00,19:54:00,20:24:00,20:54:00' WHERE route_id = 3 AND direction = '去程' AND stop_order = 10;
UPDATE bus_route_stations SET schedule = '18:13:00,18:28:00,18:48:00,18:58:00,19:18:00,19:43:00,19:58:00,20:28:00,20:58:00' WHERE route_id = 3 AND direction = '去程' AND stop_order = 11;
UPDATE bus_route_stations SET schedule = '18:19:00,18:34:00,18:54:00,19:04:00,19:24:00,19:49:00,20:04:00,20:34:00,21:04:00' WHERE route_id = 3 AND direction = '去程' AND stop_order = 12;
UPDATE bus_route_stations SET schedule = '18:20:00,18:35:00,18:55:00,19:05:00,19:25:00,19:50:00,20:05:00,20:35:00,21:05:00' WHERE route_id = 3 AND direction = '去程' AND stop_order = 13;

"\nUPDATE bus_route_stations SET schedule = '17:45:00,18:00:00,18:20:00,18:30:00,18:50:00,19:15:00,19:30:00,20:00:00,20:30:00' WHERE route_id = 3 AND direction = '去程' AND stop_name = '花蓮轉運站';\nUPDATE bus_route_stations SET schedule = '17:46:00,18:01:00,18:21:00,18:31:00,18:51:00,19:16:00,19:31:00,20:01:00,20:31:00' WHERE route_id = 3 AND direction = '去程' AND stop_name = '勤天商旅';\nUPDATE bus_route_stations SET schedule = '17:49:00,18:04:00,18:24:00,18:34:00,18:54:00,19:19:00,19:34:00,20:04:00,20:34:00' WHERE route_id = 3 AND direction = '去程' AND stop_name = '市立圖書館';\nUPDATE bus_route_stations SET schedule = '17:51:00,18:06:00,18:26:00,18:36:00,18:56:00,19:21:00,19:36:00,20:06:00,20:36:00' WHERE route_id = 3 AND direction = '去程' AND stop_name = '麗星飯店';\nUPDATE bus_route_stations SET schedule = '17:59:00,18:14:00,18:34:00,18:44:00,19:04:00,19:29:00,19:44:00,20:14:00,20:44:00' WHERE route_id = 3 AND direction = '去程' AND stop_name = '花蓮又一村文創園區';\nUPDATE bus_route_stations SET schedule = '18: